# `HumanInTheLoopMiddleware`

Middleware that pauses selected tool calls and requests a human decision before execution continues.

A reviewer can approve, edit, reject, or directly respond on behalf of a tool. Tools not included in the middleware configuration are automatically approved.

- Bases: `AgentMiddleware[StateT, ContextT, ResponseT]`

## Constructor

```python
HumanInTheLoopMiddleware(
    interrupt_on: dict[str, bool | InterruptOnConfig],
    *,
    description_prefix: str = "Tool execution requires approval"
)
```

### Parameters

* `interrupt_on` — Maps tool names to their human-review configuration.
  * `True` — Enables all four decisions: `approve`, `edit`, `reject`, and `respond`.
  * `False` — Automatically approves the tool without interrupting.
  * `InterruptOnConfig` — Defines the allowed decisions and optional custom behaviour.
  * Tools missing from this mapping are automatically approved.
* `description_prefix` — Prefix used to generate the default review description.
  * It is ignored when the tool configuration provides its own `description`.

## Attributes

* `interrupt_on` — Stores the resolved configurations for tools that require review.
  * Entries configured as `False` are removed.
  * Entries configured as `True` are converted into an `InterruptOnConfig` allowing every decision type.
* `description_prefix` — Stores the prefix used for automatically generated review descriptions.

## Methods

### 1. `after_model`

Examines the latest `AIMessage`, interrupts execution for configured tool calls, processes the human decisions, and returns the updated messages.

- **Syntax:**

```python
after_model(
    self,
    state: AgentState[Any],       # Current agent state
    runtime: Runtime[ContextT]    # Current runtime context
) -> dict[str, Any] | None
```

- **Behaviour:**
  * Finds the most recent `AIMessage` containing tool calls.
  * Selects tool calls configured for human review.
  * Applies the optional `when` predicate before interrupting.
  * Groups all selected calls into one `HITLRequest`.
  * Calls LangGraph's `interrupt()` function and waits for human decisions.
  * Preserves the original order of tool calls.
  * Adds synthetic `ToolMessage` objects for rejected or human-answered calls.
  * Returns `None` when no review is required.

- **Raises:**
  * `ValueError` — When the number of supplied decisions does not match the number of interrupted tool calls.

### 2. `aafter_model`

Asynchronous version of `after_model`.

- **Syntax:**

```python
async aafter_model(
    self,
    state: AgentState[Any],       # Current agent state
    runtime: Runtime[ContextT]    # Current runtime context
) -> dict[str, Any] | None
```

- It delegates directly to `after_model`.

### 3. `_create_action_and_config`

Creates the human-facing action request and review policy for one tool call.

- **Syntax:**

```python
_create_action_and_config(
    self,
    tool_call: ToolCall,                # Tool call proposed by the model
    config: InterruptOnConfig,          # Review configuration for the tool
    state: AgentState[Any],             # Current agent state
    runtime: Runtime[ContextT]          # Current runtime context
) -> tuple[ActionRequest, ReviewConfig]
```

- **Description selection order:**
  1. Calls `description` when it is a callable.
  2. Uses `description` directly when it is a string.
  3. Otherwise generates a description from `description_prefix`, tool name, and arguments.

### 4. `_process_decision`

Processes one human decision and returns the revised tool call and an optional synthetic tool message.

- **Syntax:**

```python
_process_decision(
    decision: Decision,             # Human decision
    tool_call: ToolCall,            # Original tool call
    config: InterruptOnConfig       # Allowed decisions for the tool
) -> tuple[ToolCall | None, ToolMessage | None]
```

- **Decision handling:**
  * `approve` — Keeps the original tool call unchanged.
  * `edit` — Replaces the tool name and arguments while preserving the original tool-call ID.
  * `reject` — Produces an error `ToolMessage` explaining that the tool was not executed.
  * `respond` — Produces a successful `ToolMessage` containing the human's response instead of executing the tool.

- **Raises:**
  * `ValueError` — When the chosen decision type is not allowed by the tool configuration.

### 5. `_should_interrupt`

Determines whether a specific tool call should trigger human review.

- **Syntax:**

```python
_should_interrupt(
    self,
    tool_call: ToolCall,
    config: InterruptOnConfig,
    state: AgentState[Any],
    runtime: Runtime[ContextT]
) -> bool
```

- Returns `True` when no `when` predicate is configured.
- Otherwise, constructs a `ToolCallRequest` and returns the predicate result.

---

# Supporting Types

## `DecisionType`

Defines the supported human decision names.

```python
DecisionType = Literal["approve", "edit", "reject", "respond"]
```

## `Action`

Represents an action selected or edited by a human reviewer.

```python
class Action(TypedDict):
    name: str
    args: dict[str, Any]
```

### Fields

* `name` — Name of the requested action or tool.
* `args` — Arguments passed to the action.

## `ActionRequest`

Represents an action presented to a human for review.

```python
class ActionRequest(TypedDict):
    name: str
    args: dict[str, Any]
    description: NotRequired[str]
```

### Fields

* `name` — Name of the requested action.
* `args` — Arguments proposed by the model.
* `description` — Optional human-readable explanation of the review request.

## `ReviewConfig`

Defines the review policy associated with an action request.

```python
class ReviewConfig(TypedDict):
    action_name: str
    allowed_decisions: list[DecisionType]
    args_schema: NotRequired[dict[str, Any]]
```

### Fields

* `action_name` — Name of the action governed by the policy.
* `allowed_decisions` — Decisions the reviewer may choose.
* `args_schema` — Optional JSON schema describing editable action arguments.

## `HITLRequest`

Payload sent through `interrupt()` for human review.

```python
class HITLRequest(TypedDict):
    action_requests: list[ActionRequest]
    review_configs: list[ReviewConfig]
```

### Fields

* `action_requests` — Actions that require human review.
* `review_configs` — Review policies corresponding to those actions.

## `HITLResponse`

Payload returned after human review.

```python
class HITLResponse(TypedDict):
    decisions: list[Decision]
```

### Fields

* `decisions` — Human decisions in the same order as the interrupted actions.

---

# Decision Types

## `ApproveDecision`

Approves the original action without modification.

```python
class ApproveDecision(TypedDict):
    type: Literal["approve"]
```

## `EditDecision`

Replaces the proposed action with an edited action.

```python
class EditDecision(TypedDict):
    type: Literal["edit"]
    edited_action: Action
```

### Fields

* `type` — Must be `"edit"`.
* `edited_action` — Updated action name and arguments.

## `RejectDecision`

Rejects the tool call and returns an error result to the model.

```python
class RejectDecision(TypedDict):
    type: Literal["reject"]
    message: NotRequired[str]
```

### Fields

* `type` — Must be `"reject"`.
* `message` — Optional explanation supplied to the model.
  * A default rejection message is generated when this field is omitted.

## `RespondDecision`

Allows the reviewer to answer on behalf of the tool without executing it.

```python
class RespondDecision(TypedDict):
    type: Literal["respond"]
    message: str
```

### Fields

* `type` — Must be `"respond"`.
* `message` — Content returned to the model in a successful synthetic `ToolMessage`.

## `Decision`

Union of every supported decision payload.

```python
Decision = (
    ApproveDecision
    | EditDecision
    | RejectDecision
    | RespondDecision
)
```

---

# `InterruptOnConfig`

Configuration for a tool that may require human review.

```python
class InterruptOnConfig(TypedDict):
    allowed_decisions: list[DecisionType]
    description: NotRequired[str | _DescriptionFactory]
    args_schema: NotRequired[dict[str, Any]]
    when: NotRequired[Callable[[ToolCallRequest], bool]]
```

## Fields

* `allowed_decisions` — Decisions permitted for the tool.
* `description` — Optional static string or callable used to describe the review request.
* `args_schema` — Optional JSON schema for editable action arguments.
* `when` — Optional predicate that determines whether an individual tool call should interrupt.
  * Returning `True` requests human review.
  * Returning `False` automatically approves that call.

## Dynamic Description Signature

```python
def description(
    tool_call: ToolCall,
    state: AgentState[Any],
    runtime: Runtime[ContextT]
) -> str:
    ...
```

## Conditional Interrupt Signature

```python
def when(request: ToolCallRequest) -> bool:
    ...
```

---

# Usage Examples

## Review every call to a tool

```python
from langchain.agents.middleware import HumanInTheLoopMiddleware

middleware = HumanInTheLoopMiddleware(
    interrupt_on={
        "delete_file": True,
        "send_email": True,
    }
)
```

Both tools allow `approve`, `edit`, `reject`, and `respond` decisions.

## Allow only approval or rejection

```python
middleware = HumanInTheLoopMiddleware(
    interrupt_on={
        "delete_file": {
            "allowed_decisions": ["approve", "reject"],
            "description": "Review this file deletion request.",
        }
    }
)
```

## Interrupt conditionally

```python
middleware = HumanInTheLoopMiddleware(
    interrupt_on={
        "delete_file": {
            "allowed_decisions": ["approve", "reject"],
            "when": lambda request: request.tool_call["args"]
            .get("path", "")
            .startswith("/etc"),
        }
    }
)
```

Only deletion requests targeting paths under `/etc` require human approval.

## Generate a dynamic description

```python
import json


def describe_tool_call(tool_call, state, runtime):
    return (
        f"Tool: {tool_call['name']}\n"
        f"Arguments:\n{json.dumps(tool_call['args'], indent=2)}"
    )


middleware = HumanInTheLoopMiddleware(
    interrupt_on={
        "send_email": {
            "allowed_decisions": ["approve", "edit", "reject"],
            "description": describe_tool_call,
        }
    }
)
```

## Automatically approve a tool

```python
middleware = HumanInTheLoopMiddleware(
    interrupt_on={
        "delete_file": True,
        "read_file": False,
    }
)
```

`delete_file` requires review, while `read_file` proceeds automatically.

## Human response on behalf of a tool

A reviewer can return a direct response for an ask-user-style tool:

```python
{
    "type": "respond",
    "message": "The preferred deployment region is Mumbai."
}
```

The real tool is skipped, and the message is returned to the model as a successful `ToolMessage`.

---

# Important Notes

* Human-in-the-loop execution depends on LangGraph interrupt and persistence support so execution can pause and later resume.
* Decisions must be returned in the same order as the interrupted action requests.
* A decision is valid only when its type appears in the corresponding tool's `allowed_decisions` list.
* Tool calls without a matching `interrupt_on` entry proceed without review.
* In this source version, `args_schema` is defined by the configuration types, but `_create_action_and_config` currently builds `ReviewConfig` using only `action_name` and `allowed_decisions`.